<!-- nav -->
[← 8 · From rules to code](08_From_Rules_To_Code.ipynb) · [Home](../../README.md) · [Programs design →](../../docs/PROGRAMS.md)

# 9 · Rules in hardware

**The problem.** The cold chain's sensor board is an FPGA, and its team writes SystemVerilog. The rules kept as data
with mbse-expressions, "the reading stays between 2.0 and 8.0 °C" (in tenths of a degree, as the sensor counts) and
"a shipment is fresh for 72 hours", must hold in the testbench's random stimulus and be checked in simulation. And the
RTL already checks things that nobody has written down as rules.

Verilog is the fourth language here: Verilog and SystemVerilog as one tree language, whose kinds follow IEEE 1800's
grammar, read by slang. A second **bridge** carries mbse-expressions' SystemVerilog dialect to and from its trees, as
case study 8's did for Python.

In [1]:
import { Evaluators, Expressions as E } from "@mbse/expressions";
import { Domains, Evaluators as SvEvaluators, Expressions as SvDialect, Text } from "@mbse/expressions/Dialects/SystemVerilog";
import * as Translators from "@mbse/expressions/Translators";
import { Verilog as Bridge } from "@mbse/programs/Bridges";
import { Errors } from "@mbse/programs/Framework";
import { walk } from "@mbse/programs/Framework/Syntax";
import { SystemVerilog2023, Syntax as S, Verilog2005 } from "@mbse/programs/Verilog";
import { Proxies, Schemas } from "@mbse/schemas/Framework";

const Shipment = new Schemas.OfObject.Builder().name("Shipment").ref()
  .properties((p: any) => p.name("deci_celsius").of((t: any) => t.as_native(BigInt)),
    (p: any) => p.name("hours").of((t: any) => t.as_native(BigInt))).create();
const store: any = new Proxies.OfStore();
store.register(Shipment);
const self = E.variable("this");
const rules: Record<string, any> = {
  in_range: E.literal(20n).le(self.deci_celsius).and_(self.deci_celsius.le(80n)).data,
  fresh: self.hours.le(72n).data,
};

## Step 1: SystemVerilog as a tree

The testbench's transaction is a class of random variables. It parses into a tree like any other program, by the
standard it's written to; a class is SystemVerilog's, so Verilog-2005 refuses to print it, at its path:

In [2]:
const reading: any = SystemVerilog2023.parse(`class reading;
    rand int deci_celsius;  // tenths of a degree
    rand int hours;
endclass
`);
console.log(reading.items[0].items.map((item: any) => item.constructor.name));
try {
  Verilog2005.print(reading);
} catch (error) {
  if (!(error instanceof Errors.PrintError)) throw error;
  console.log(error.message);
}

[ 'VariableDeclaration', 'Comment', 'VariableDeclaration' ]
items[0]: ClassDeclaration is not Verilog, but this is Verilog-2005


## Step 2: rules become constraints

mbse-expressions translates each rule from its neutral form (Basic) to its SystemVerilog dialect; the bridge turns
that into a constraint, which joins the class's items. Randomization now draws only readings the rules allow. `this`
is the rule's subject, and in a class it's SystemVerilog's own `this`:

In [3]:
const toSv = Translators.between(E.DIALECT, SvDialect.DIALECT);
for (const [name, rule] of Object.entries(rules)) {
  reading.items[0].items.push(Bridge.constraint(`c_${name}`, toSv.forward(rule)));
}
console.log(SystemVerilog2023.print(reading));

class reading;
    rand int deci_celsius;  // tenths of a degree
    rand int hours;

    constraint c_in_range {
        20 <= this.deci_celsius && this.deci_celsius <= 80;
    }

    constraint c_fresh {
        this.hours <= 72;
    }
endclass



## Step 3: rules become checks

The same rules become methods the testbench calls to check a transaction, and an immediate assertion that reports
where a simulation breaks them. Evaluated by the SystemVerilog dialect's rules, they agree with Basic on each shipment:

In [4]:
console.log(SystemVerilog2023.print(Bridge.function_("in_range", [], toSv.forward(rules.in_range))));
console.log(SystemVerilog2023.print(Bridge.assertion(toSv.forward(rules.fresh), "stale shipment")));
const shipments = [store.Shipment().deci_celsius(55n).hours(30n).create(), store.Shipment().deci_celsius(91n).hours(80n).create()];
for (const shipment of shipments) {
  const inHardware = Object.values(rules).map((rule) => String(SvEvaluators.OfAny(toSv.forward(rule), { this: shipment })));
  console.log(inHardware, Object.values(rules).map((rule) => Evaluators.predicate(rule, shipment)));
}

function automatic logic in_range;
    return 20 <= this.deci_celsius && this.deci_celsius <= 80;
endfunction
assert (this.hours <= 72) else $error("stale shipment");
[ "1'b1", "1'b1" ] [ true, true ]
[ "1'b0", "1'b0" ] [ false, false ]


## Step 4: an RTL assertion becomes a rule

The FIFO's monitor already asserts something. The bridge reads it into a rule, which can now be stored, reviewed and
evaluated beside the others. Evaluated by IEEE 1800's rules, it sees four states: an unknown bit makes the check
unknown (`1'bx`), as in simulation, rather than true or false.

In [5]:
const monitor = SystemVerilog2023.parse(`module fifo_monitor(input logic clk, input logic [7:0] count, input logic overflow);
    always @(posedge clk) begin
        assert (count <= 8'd200 && !overflow) else $error("fifo too full");
    end
endmodule
`);
const assertion = [...walk(monitor)].find((node) => node instanceof S.ImmediateAssertion) as S.ImmediateAssertion;
const rule = Bridge.term_of_assertion(assertion);
console.log(Text.ToText(rule));
for (const [count, overflow] of [["00010000", "0"], ["11110000", "0"], ["00010000", "x"], ["1111xxxx", "0"]] as const) {
  const value = SvEvaluators.OfAny(rule, { count: Domains.Logic.of(count), overflow: Domains.Logic.of(overflow) });
  console.log(count, overflow, String(value));
}

count <= 8'd200 && !overflow
00010000 0 1'b1
11110000 0 1'b0
00010000 x 1'bx
1111xxxx 0 1'bx


## Step 5: what a rule can't hold

A rule is smaller than a design: pure, with no statements, no timing, and only the dialect's expressions. An indexed
part-select, `data[3+:2]`, has no counterpart, and the bridge says where:

In [6]:
const check = SystemVerilog2023.parse("function automatic logic f(input logic [7:0] data); return data[3+:2] == 2'b01; endfunction\n");
try {
  Bridge.term_of_function(check.items[0] as S.FunctionDeclaration);
} catch (error) {
  if (!(error instanceof Errors.TranspileError)) throw error;
  console.log(error.message);
}

body[0].value.left: a part-select with +: has no counterpart in the dialect


## Why it works this way

- **One rule, every target.** The rule written once as data is Python in the pipeline (case study 8), a constraint in
  the testbench and an assertion in simulation here, each printed by its language's own printer. Nobody copies it by
  hand, so nobody copies it wrong.
- **Four states, not two.** The SystemVerilog dialect evaluates as IEEE 1800 does, with `x` and `z`, so a rule read
  from RTL means there what it means in simulation.
- **Read by the reference front end.** Verilog is read by slang, in both implementations, so the trees and the rules
  read from them are the same in Python and TypeScript.
- **Refusing is part of the contract.** What a rule can't hold fails at a path, never approximated.

<!-- nav -->
[← 8 · From rules to code](08_From_Rules_To_Code.ipynb) · [Home](../../README.md) · [Programs design →](../../docs/PROGRAMS.md)